# 1. From a Task to a Flow

Every calculation in AbiPy is, at bottom, an `AbinitTask`: one `AbinitInput`
plus a working directory. `Task`s can depend on each other (e.g. a
non-self-consistent run needs the density from a self-consistent one), and
a `Flow` is what takes care of that bookkeeping for you -- ordering tasks
correctly, and letting you (re)launch the whole thing with one command.

This notebook builds the same calculation -- the band structure of silicon
-- three times, in increasingly automated ways, using the standalone
scripts in `../Examples/`:

1. `run_si_gstate.py` -- a single Task (ground state).
2. `run_si_nscf.py` -- a second Task, depending on the first through a
   **hardcoded path**.
3. `run_si_ebands.py` -- both tasks registered in a `Flow`, with the
   dependency expressed **through the task object itself**.

Each has already been run for you; we'll look at the script, then the
result, at each step.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from abipy import abilab
import abipy.flowtk as flowtk
abilab.enable_notebook()

%matplotlib inline

## Step 1: a single Task

The simplest possible unit of work: one `AbinitInput` wrapped in one
`AbinitTask`, built and run directly -- no `Flow`, no `Work`.

In [ ]:
print(open("../Examples/run_si_gstate.py").read())

Run from a terminal, from inside `Examples/` (so the output directory lands
there, next to the script):

```
cd ../Examples
python run_si_gstate.py
```

This produces `task_si_gstate/`, including the density (`out_DEN.nc`) that
Step 2 depends on.

In [ ]:
with abilab.abiopen("../Examples/task_si_gstate/outdata/out_GSR.nc") as gsr:
    print("energy:", gsr.energy, "eV")
    print(gsr.ebands)

## Step 2: a second, dependent Task

Same idea, but this task's input is a band-structure (NSCF) calculation,
and it needs the density file produced in Step 1.

In [ ]:
print(open("../Examples/run_si_nscf.py").read())

Notice the dependency: `deps = {density: 'DEN'}`, where `density` is just
the **string path** `'task_si_gstate/outdata/out_DEN.nc'`. This works, but
it's fragile -- rename or move `task_si_gstate/`, and this breaks with no
warning until Abinit complains about a missing file.

```
python run_si_nscf.py
```

In [ ]:
with abilab.abiopen("../Examples/task_si_nscf/outdata/out_GSR.nc") as gsr:
    ebands = gsr.ebands

fig = ebands.plot(color="b", show=False)
fig.gca().set_ylim(-12.5, 7.5);

## Step 3: wrapping both Tasks in a Flow

Same two tasks, same physics -- but now registered together in a `Work`
inside a `Flow`, with the dependency expressed as `deps={gs_task: 'DEN'}`:
a reference to the *task object* itself, not a path.

In [ ]:
print(open("../Examples/run_si_ebands.py").read())

```
python run_si_ebands.py
```

The `Flow` now knows the dependency graph, and can be inspected, resumed,
or re-run reliably, however many tasks it grows to:

In [ ]:
flow = flowtk.Flow.from_file("../Examples/flow_si_ebands")
flow.get_graphviz()

In [ ]:
task = flow[0][1]  # second task of the first Work: the NSCF run
gsr_path = task.outdir.has_abiext("GSR")

with abilab.abiopen(gsr_path) as gsr:
    ebands = gsr.ebands

fig = ebands.plot(color="b", show=False)
fig.gca().set_ylim(-12.5, 7.5);

Same plot as Step 2 -- same physics, computed with the same two Abinit
runs, just orchestrated by AbiPy instead of two manual scripts and a
hardcoded path.

> **Exercise.** Silicon's fundamental gap is indirect: the conduction band
> minimum is not at $\Gamma$. Zoom in on the plot above (or replot with a
> narrower `set_ylim`) to see it. We'll come back to this in
> `2.3-BandStructure.ipynb`, comparing with GaAs -- which has a *direct*
> gap at $\Gamma$.

Every flow from here on (Notebook 2 onward) is expressed as a `Flow` from
the start, for exactly this reason: it scales past two tasks without
becoming unmanageable.

Continue with [`2-Existing_flows.ipynb`](2-Existing_flows.ipynb).